# 07 — Geologic Hazards

**Series:** Tribal Soils and Geology

---

## Geologic Hazards on Pine Ridge and Rosebud

Geologic hazards are natural physical processes that threaten life,
infrastructure, or land use. On Pine Ridge and Rosebud, three hazards
are geologically significant and directly traceable to the Pierre Shale
stratigraphy that dominates the study area:

**Expansive soils and rock:** Smectite clay in the Pierre Shale and its
derived soils swells when wetted and shrinks when dried. Volume changes
of 30-40% are documented. This is the single most pervasive geologic
hazard on both reservations — affecting roads, foundations, pipelines,
and water infrastructure.

**Landslides and slope instability:** The Pierre Shale contains weak,
plastic clay layers that act as failure planes when saturated. Landslides
are common on valley walls throughout the White River system and along
the Pine Ridge Escarpment. Colluvial deposits at valley margins are
frequently unstable.

**Radon:** The Pierre Shale and some of the underlying formations contain
uranium-bearing minerals that produce radon gas. Interior housing on
both reservations warrants radon testing.

These hazards disproportionately affect Tribal communities because
infrastructure investment on Pine Ridge and Rosebud has historically
been insufficient to engineer around known geologic constraints.

In [ ]:
import sys
from pathlib import Path
REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path: sys.path.insert(0, str(REPO_ROOT))
import warnings, numpy as np, pandas as pd
import geopandas as gpd, matplotlib.pyplot as plt
import matplotlib.patches as mpatches, contextily as ctx, yaml
from shapely.geometry import box as sbox
from src.constants import (
    CRS_GEOGRAPHIC, CRS_PROJECTED, CRS_WEB, REPO_ROOT as _REPO_ROOT,
    OUTPUTS_DIR, FIGURES_DIR, PINE_RIDGE_BBOX, ROSEBUD_BBOX,
    COMBINED_BBOX, STUDY_BBOX, WSD_3D_MODEL, WSD_KEY_UNITS,
)
from src.loaders import load_tribal_boundaries, load_usgs_well_sites
from src.sovereignty import print_data_acknowledgment, generate_citations
warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline
with open(_REPO_ROOT / "config" / "config.yaml") as f: CONFIG = yaml.safe_load(f)
TEAL="#007A6E"; TEAL_LT="#E0F4F2"; GRAY="#566573"; TERRACOTTA="#C0392B"
def despine(ax):
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
primary = load_tribal_boundaries(["Pine Ridge","Rosebud"])
print(f"Ready. Nations: {len(primary)}")

In [ ]:
print_data_acknowledgment(source_keys=["usgs_state_geology","usgs_landslide"])

---
## 1. Expansive Soil Mapping from SSURGO

In [ ]:
from src.loaders import load_ssurgo_mapunits, load_ssurgo_components, load_ssurgo_horizons

mapunits  = load_ssurgo_mapunits()
components = load_ssurgo_components()
horizons   = load_ssurgo_horizons()

if not horizons.empty and "lep_r" in horizons.columns:
    horizons["lep_r"] = pd.to_numeric(horizons["lep_r"], errors="coerce")
    lep_threshold = CONFIG["ssurgo"]["lep_high"] * 100   # convert to percent

    # Component-level max LEP
    if "cokey" in horizons.columns and "cokey" in components.columns:
        comp_max_lep = horizons.groupby("cokey")["lep_r"].max().reset_index()
        comp_max_lep.columns = ["cokey","max_lep"]
        components_lep = components.merge(comp_max_lep, on="cokey", how="left")

        # Map unit max LEP (dominant component)
        if "mukey" in components_lep.columns:
            mu_max_lep = components_lep.groupby("mukey")["max_lep"].max().reset_index()
            mapunits_hazard = mapunits.merge(mu_max_lep, on="mukey", how="left")
            mapunits_hazard["expansive_class"] = pd.cut(
                mapunits_hazard["max_lep"],
                bins=[0, 3, 6, 9, 100],
                labels=["Low","Moderate","High","Very High"]
            )
            print("EXPANSIVE SOIL HAZARD BY MAP UNIT")
            print("=" * 50)
            dist = mapunits_hazard["expansive_class"].value_counts()
            for cls, n in dist.items():
                print(f"  {cls:<15}: {n:>5} map units")
else:
    mapunits_hazard = mapunits
    print("LEP data not available. Load SSURGO horizons to compute expansive soil hazard.")

In [ ]:
if "expansive_class" in (mapunits_hazard.columns if "mapunits_hazard" in dir() else []):
    HAZARD_COLORS = {
        "Low":       "#27AE60",
        "Moderate":  "#F39C12",
        "High":      "#E74C3C",
        "Very High": "#7B241C",
    }
    fig, axes = plt.subplots(1, 2, figsize=(15, 8))
    for ax, (nation, bbox_k) in zip(axes,
        [("Pine Ridge", PINE_RIDGE_BBOX), ("Rosebud", ROSEBUD_BBOX)]):
        clip_b = sbox(*bbox_k)
        sub    = mapunits_hazard[mapunits_hazard.geometry.intersects(clip_b)].copy()
        nb_gdf = primary[primary["common_name"].str.contains(nation.split()[0])]

        for cls, color in HAZARD_COLORS.items():
            cls_sub = sub[sub["expansive_class"] == cls]
            if not cls_sub.empty:
                cls_sub.to_crs(CRS_WEB).plot(
                    ax=ax, color=color, alpha=0.8, linewidth=0,
                    label=f"{cls} ({len(cls_sub)})"
                )
        if not nb_gdf.empty:
            nb_gdf.to_crs(CRS_WEB).plot(
                ax=ax, facecolor="none", edgecolor=TEAL, linewidth=2.5, zorder=5
            )
        try:
            ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, alpha=0.2, zoom=9)
        except Exception: pass
        ax.set_axis_off()
        ax.legend(fontsize=7, loc="lower right", framealpha=0.9,
                  title="Expansive Soil Hazard")
        ax.set_title(f"Expansive Soil Hazard — {nation}\n"
                     f"(SSURGO LEP-based classification)",
                     fontsize=10, fontweight="bold")

    plt.suptitle("Expansive Soil Hazard — Pine Ridge and Rosebud Reservations\n"
                 "Based on SSURGO Linear Extensibility Percent (LEP)",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "07_expansive_soil_hazard.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Compute expansive soil hazard first (cell above).")

---
## 2. Landslide Susceptibility

In [ ]:
# Slope-based landslide susceptibility proxy
# USGS landslide hazard API (national dataset)
# Alternative: compute from SSURGO + DEM slope

import requests

print("Querying USGS landslide susceptibility data...")
r = requests.get(
    "https://landslides.usgs.gov/hazus/geoserver/ows",
    params={
        "service":     "WFS",
        "version":     "1.0.0",
        "request":     "GetFeature",
        "typeName":    "hazus:ls_susceptibility",
        "bbox":        f"{COMBINED_BBOX[0]},{COMBINED_BBOX[1]},"
                       f"{COMBINED_BBOX[2]},{COMBINED_BBOX[3]}",
        "outputFormat":"application/json",
        "maxFeatures": "1000",
    },
    timeout=60,
)
print(f"Status: {r.status_code}")
if r.status_code == 200 and len(r.content) > 100:
    try:
        ls = gpd.read_file(r.content)
        print(f"Landslide susceptibility features: {len(ls)}")
        if not ls.empty:
            print(ls.columns.tolist())
    except Exception as e:
        print(f"Could not parse response: {e}")
else:
    print("Landslide susceptibility service not available.")
    print("Alternative: use SSURGO slope gradient + pierre shale indicator")
    print("  from notebooks 03 and 05 to derive a local proxy index.")

---
## 3. Pierre Shale Depth as Hazard Proxy

Where the 3D model is available, depth to the Pierre Shale top provides
a direct proxy for both expansive soil hazard and slope instability risk.
Shallow Pierre Shale (< 5m) means the expansive clay is in the active
zone of moisture fluctuation — where shrink-swell is most severe.

This analysis requires the WSouthDakota3D.gdb from notebook 04.
When that data is available, run the following to produce a depth-to-Pierre-Shale
map over Tribal boundaries.

In [ ]:
from src.loaders import load_wsd_horizon_raster
import rasterio
import numpy as np

PIERRE_SHALE_UNIT = "Pierre Shale"   # exact name in DescriptionOfModelUnits

print(f"Attempting to load Pierre Shale horizon raster...")
pierre_ds = load_wsd_horizon_raster(PIERRE_SHALE_UNIT)

if pierre_ds is None:
    print()
    print("Pierre Shale horizon raster not available.")
    print("Download WSouthDakota3D.gdb from https://doi.org/10.5066/P9LK4QHJ")
    print("and place in data/raw/geology/")
    print()
    print("When available, this cell will produce:")
    print("  - Depth-to-Pierre-Shale raster clipped to Tribal boundaries")
    print("  - Map showing areas where Pierre Shale < 5m depth (highest hazard)")
    print("  - Summary statistics by reservation")
else:
    data = pierre_ds.read(1, masked=True)
    print(f"Pierre Shale horizon loaded.")
    print(f"  Elevation range: {data.min():.1f} to {data.max():.1f} m")
    print(f"  Shape: {data.shape}")
    print()
    print("Next step: subtract from DEM to get depth.")
    print("Use rasterio + USGS 3DEP elevation service for DEM.")
    pierre_ds.close()

---
## 4. Infrastructure Implications

In [ ]:
# Summary table: hazard type, mechanism, affected infrastructure, mitigation
hazard_summary = [
    ("Expansive soils",
     "Pierre Shale smectite clay swells when wet, shrinks dry",
     "Roads, foundations, pipelines, sewers, water lines",
     "Lime stabilization, engineered fill, flexible pavements, deep foundations"),
    ("Slope instability",
     "Saturated Pierre Shale fails on slopes as gentle as 3-5 degrees",
     "Roads on valley walls, buildings on slopes, utility corridors",
     "Avoid sensitive slopes; drainage management; geotechnical investigation before siting"),
    ("Radon",
     "Uranium in Pierre Shale and underlying formations produces radon",
     "Interior housing, especially basements and manufactured homes",
     "EPA Zone 1 county: test all homes; radon mitigation systems"),
    ("Flash flooding",
     "Low permeability Pierre Shale generates high runoff; "
     "White River and tributaries respond rapidly",
     "Low-lying structures, bridges, roads in valley bottoms",
     "Flood plain mapping; infrastructure setback; culvert sizing"),
]

print("GEOLOGIC HAZARD SUMMARY — Pine Ridge and Rosebud")
print("=" * 70)
for hazard, mechanism, infrastructure, mitigation in hazard_summary:
    print(f"\n{hazard.upper()}")
    print(f"  Mechanism : {mechanism}")
    print(f"  Affected  : {infrastructure}")
    print(f"  Mitigation: {mitigation}")

print()
print("These hazards are well-characterized in published geologic literature.")
print("They are not new discoveries. Infrastructure built on Pierre Shale")
print("without geotechnical engineering fails predictably. The disproportionate")
print("impact on Tribal communities reflects historical underinvestment, not")
print("geological inevitability.")

In [ ]:
print(generate_citations(["usgs_state_geology","usgs_landslide","usgs_3d_model"]))